<a href="https://colab.research.google.com/github/Arxivory/Urban-Heat-Safety-PINN/blob/main/urban_heat_safety_pinn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Urban Heat Safety: A Physics-Informed Neural Network (PINN) Approach**
### *Modeling Microclimate Heat Stress using Steady-State Thermodynamics*

**Author:** Arxivory  
**Project Scope:** This research implements a PINN to predict the **Wet Bulb Globe Temperature (WBGT)**, a composite temperature used to estimate the effect of temperature, humidity, wind speed, and solar radiation on humans. Unlike standard "black-box" machine learning, this model is constrained by the **Steady-State Energy Balance Equation** to ensure physical consistency.

# **Physics Setup**

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

**Physical Constants**

In [ ]:
SIGMA = 5.670373e-8
EPS_G = 0.95
ALBEDO_G = 0.05
D = 0.15

**Physics Computer**

To generate ground-truth data, we solve for the temperature at which a black globe thermometer reaches thermal equilibrium. This state is defined by the balance of radiative and convective heat fluxes:$$Q_{shortwave} + Q_{longwave} - Q_{convection} = 0$$
Where:
*   Shortwave Radiation ($Q_{sw}$): Absorbed solar energy, defined by $(1 - \alpha)
\cdot I$, where $\alpha$ is albedo and $I$ is solar intensity.

*   Longwave Radiation ($Q_{lw}$): Heat exchange between the globe and the environment, governed by the Stefan-Boltzmann Law: $\epsilon \sigma (T_{air}^4 - T_g^4)$.

*   Convective Heat ($Q_{conv}$): Heat loss to the surrounding air, defined by $h(v) \cdot (T_g - T_{air})$, where $h(v)$ is the convective heat transfer coefficient dependent on wind velocity $v$.

In [ ]:
def calculate_physics_tg(Ta, wind_speed, solar_rad):
  Ta_k = Ta + 273.15
  Tg_k = Ta_k + 5.0

  for _ in range(100):
    h = 0.315 * (wind_speed**0.58) / (D**0.42)

    q_solar = (1 - ALBEDO_G) * solar_rad
    q_net_longwave = EPS_G * SIGMA * (Ta_k**4 - Tg_k**4)
    q_conv = h * (Tg_k - Ta_k)

    residual = q_solar + q_net_longwave - q_conv

    Tg_k += residual * 0.01

  return Tg_k - 273.15

**Synthetic Data Generation (for now)**

In [ ]:
np.random.seed(42)
n_samples = 1000

data = {
    'air_temp': np.random.uniform(20, 45, n_samples),
    'wind_speed': np.random.uniform(0.5, 10, n_samples),
    'solar_rad': np.random.uniform(0, 1000, n_samples)
}

df = pd.DataFrame(data)
df['target_tg'] = df.apply(lambda row: calculate_physics_tg(row['air_temp'],
                                                            row['wind_speed'],
                                                            row['solar_rad']), axis=1)
print("Sample of generated physics data: ")
print(df.head())


**Visualization of the Synthetic Data**

In [ ]:
plt.scatter(df['air_temp'], df['target_tg'], c=df['solar_rad'], cmap='hot', alpha=0.5)
plt.colorbar(label='Solar Radiation (W/m^2)')
plt.xlabel('Air Temp (°C)')
plt.ylabel('Globe Temp (°C)')
plt.title('Physics-Simulated Globe Temperatures')
plt.show()

# **Neural Network Architecture and Data Preparation**

Our model is a Multi-Layer Perceptron (MLP). Because variables like Solar Radiation ($0–1000 W/m^2$) and Wind Speed ($0.5–10 m/s$) exist on different scales, we apply Min-Max Scaling to map all inputs and outputs to the range $[0, 1]$:$$x' = \frac{x - \text{min}(x)}{\text{max}(x) - \text{min}(x)}$$Architecture:
* Input Layer: 3 nodes ($T_{air}, v, I$).

* Hidden Layers: Two layers of 32 neurons each with ReLU (Rectified Linear Unit) activation functions: $f(x) = \max(0, x)$.

* Output Layer: 1 node (Predicted $T_g$).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

**Data Preparation**

In [ ]:
X = df[['air_temp', 'wind_speed', 'solar_rad']].values
y = df['target_tg'].values.reshape(-1, 1)

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X_scaled,
                                                    y_scaled,
                                                    test_size=0.2,
                                                    random_state=42)
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test)

**Neural Network Model**

In [ ]:
import torch.nn as nn

class HeatSafetyNet(nn.Module):
  def __init__(self):
    super(HeatSafetyNet, self).__init__()
    self.layers = nn.Sequential(
        nn.Linear(3, 32),
        nn.ReLU(),
        nn.Linear(32, 32),
        nn.ReLU(),
        nn.Linear(32, 1)
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
model = HeatSafetyNet()
print(model)

# **Physics-Informed Loss Function**

The unique advantage of a Physics-Informed Neural Network is its composite loss function. The model is optimized not just to match the data, but to respect the laws of physics:$$\mathcal{L}_{total} = \mathcal{L}_{MSE} + \lambda \mathcal{L}_{physics}$$
1. Supervised Loss ($\mathcal{L}_{MSE}$): The standard Mean Squared Error between the predicted $\hat{T}_g$ and the ground truth $T_g$.

2. Physics Loss ($\mathcal{L}_{physics}$): The residual of the energy balance equation. We take the model's output, unscale it, and calculate the deviation from equilibrium.

3. $\lambda$ (Lambda): A weighting factor (set to 0.1) that balances the importance of the physical constraint against the training data.

**The Function**

In [ ]:
def physics_loss_func(inputs, outputs, scaler_X, scaler_y):
  inputs_unscaled = inputs * torch.tensor(scaler_X.data_range_) + torch.tensor(scaler_X.data_min_)
  Ta = inputs_unscaled[:, 0] + 273.15
  v = inputs_unscaled[:, 1]
  I = inputs_unscaled[:, 2]

  Tg = outputs.squeeze() * torch.tensor(scaler_y.data_range_) + torch.tensor(scaler_y.data_min_)
  Tg_k = Tg + 273.15

  sigma = 5.670373e-8
  eps_g = 0.95
  albedo_g = 0.05
  D = 0.15

  h = 0.315 * (v**0.58) / (D**0.42)

  q_solar = (1 - albedo_g) * I
  q_net_longwave = eps_g * sigma * (Ta**4 - Tg_k **4)
  q_conv = h * (Tg_k - Ta)

  residual = q_solar + q_net_longwave - q_conv

  return torch.mean(residual**2)

**Training Loop**

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
mse_criterion = nn.MSELoss()
lambda_phys = 0.1
epochs = 100

history = []

for epoch in range(epochs):
  model.train()
  optimizer.zero_grad()

  predictions = model(X_train_t)

  loss_data = mse_criterion(predictions, y_train_t)

  loss_phys = physics_loss_func(X_train_t, predictions, scaler_X, scaler_y)

  total_loss = loss_data + (lambda_phys * loss_phys)

  total_loss.backward()
  optimizer.step()

  if (epoch + 1) % 10 == 0:
    print(f"Epoch [{epoch+1}/{epochs}], Total Loss: {total_loss.item():.6f}, Physics Loss: {loss_phys.item():.6f}")

  history.append(total_loss.item())

plt.plot(history)
plt.title("Training Progress (Total Loss)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

# **Heat Safety Dashboard**

The final WBGT is calculated by integrating the model's predicted Globe Temperature ($T_g$) with the Natural Wet Bulb Temperature ($T_{nw}$), estimated via Stull’s Equation:$$WBGT = 0.7 T_{nw} + 0.2 T_g + 0.1 T_{air}$$The resulting value is mapped to color-coded safety flags (e.g., Extreme/Black Flag if $WBGT > 32.2^\circ C$) to provide actionable safety guidance for urban environments.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

def get_safety_flag(wbgt):
    """Categorizes risk based on international heat safety standards."""
    if wbgt < 26.7:
        return ("LOW (White Flag)", "#ffffff", "#000000")
    elif 26.7 <= wbgt < 29.4:
        return ("MODERATE (Green Flag)", "#2ecc71", "#ffffff")
    elif 29.4 <= wbgt < 31.1:
        return ("HIGH (Yellow Flag)", "#f1c40f", "#000000")
    elif 31.1 <= wbgt < 32.2:
        return ("VERY HIGH (Red Flag)", "#e67e22", "#ffffff")
    else:
        return ("EXTREME (Black Flag)", "#2c3e50", "#ffffff")

def estimate_tnw(td, rh):
    """Simplified Stull formula for Wet Bulb Temperature."""
    tw = td * np.arctan(0.151977 * (rh + 8.313659)**0.5) + \
         np.arctan(td + rh) - np.arctan(rh - 1.676331) + \
         0.00391838 * (rh)**1.5 * np.arctan(0.023101 * rh) - 4.686035
    return tw

def predict_safety(air_temp, wind, solar, humidity):
    model.eval()
    with torch.no_grad():
        input_data = np.array([[air_temp, wind, solar]])
        input_scaled = scaler_X.transform(input_data)
        input_t = torch.FloatTensor(input_scaled)

        pred_tg_scaled = model(input_t)

        tg = scaler_y.inverse_transform(pred_tg_scaled.numpy())[0][0]

        tnw = estimate_tnw(air_temp, humidity)
        wbgt = (0.7 * tnw) + (0.2 * tg) + (0.1 * air_temp)

        status, bg_color, text_color = get_safety_flag(wbgt)

        display_html = f"""
        <div style="padding: 20px; border-radius: 10px; background-color: {bg_color}; color: {text_color}; text-align: center; font-family: sans-serif;">
            <h2 style="margin: 0;">Predicted WBGT: {wbgt:.1f}°C</h2>
            <p style="font-size: 1.2em; font-weight: bold;">Safety Status: {status}</p>
            <hr style="border-color: {text_color}; opacity: 0.3;">
            <p>Simulated Globe Temp (Tg): {tg:.1f}°C | Air Temp: {air_temp}°C</p>
        </div>
        """
        display(HTML(display_html))

widgets.interact(predict_safety,
    air_temp=widgets.IntSlider(min=20, max=50, step=1, value=30, description='Air Temp (°C)'),
    wind=widgets.FloatSlider(min=0.5, max=10.0, step=0.1, value=2.0, description='Wind (m/s)'),
    solar=widgets.IntSlider(min=0, max=1000, step=50, value=800, description='Solar (W/m²)'),
    humidity=widgets.IntSlider(min=0, max=100, step=5, value=60, description='Humidity (%)')
);